# 🧩 Mini-Lab: First Win — Triage Your Own Text

**Module 1: Ship a Working AI Feature Today** | **Type: Mini-Lab**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Run** a triage on your own text with no install, no account, and no API key
2. **Read** a structured JSON result with `id`, `label`, `summary`, and `latency_ms`
3. **Recognize** that a plausible mock label is not always a correct label
4. **Map** the cells you just ran onto the six-box pipeline (ingest, prepare, call, validate, log, evaluate)

## Target Concepts

| Concept | Gloss |
|---------|-------|
| AI Pipeline Stages | Ingest, prepare, call, validate, log, evaluate; model is one box |
| Mock inference | A local stand-in for a model call: no API key, no network, and occasionally wrong |
| Structured JSON | A result with a stable shape: `id`, `label`, `summary`, `latency_ms` |


## Why This Matters

Mira is handed 4,000 unread support messages on a Friday. Leadership wants them sorted by morning: billing, bugs, account issues, feature requests, everything else.

You will not solve 4,000 rows today. You will do something better: **drive the whole path from "text goes in" to "label comes out" on a message you choose**, in the browser, with nothing to install.

**Privacy:** paste sample text, or personal-but-harmless text only. Never confidential or customer data — even into this local mock. The mock does not leave your session, but the habit starts now.

If this notebook will not open, download the `.ipynb` and run it in any Python 3 Jupyter environment. There is no account and no key.

Run the cells **top to bottom**. A later cell uses names defined above it.

In [1]:
import json
import time

from IPython.display import display, Markdown


def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## A Stand-In for a Model Call

`classify_message` looks like an AI call: text in, structured result out. It is not a model. It is a **mock** — a keyword function that returns a plausible label so you can run the rest of the work with zero setup.

Allowed labels: `billing`, `bug`, `account`, `feature`, `other`.

In [2]:
LABELS = ("billing", "bug", "account", "feature", "other")

KEYWORD_RULES = (
    ("billing", ("invoice", "refund", "charged", "payment", "receipt", "subscription")),
    ("bug", ("crash", "error", "broken", "bug", "not working")),
    ("account", ("password", "login", "sign in", "locked out", "username")),
    ("feature", ("would be nice", "feature", "can you add", "wishlist")),
)


def mock_infer(text):
    """Return a plausible label and a short summary. Not always correct."""
    lowered = text.lower()
    label = "other"
    for name, keywords in KEYWORD_RULES:
        if any(keyword in lowered for keyword in keywords):
            label = name
            break
    summary = " ".join(text.split())
    if len(summary) > 100:
        summary = summary[:97] + "..."
    return {"label": label, "summary": summary}


def classify_message(text, msg_id="local-1"):
    """Take one message, call the mock, check the keys, return a JSON-ready dict."""
    started = time.perf_counter()
    cleaned = (text or "").strip()
    raw = mock_infer(cleaned)
    result = {
        "id": msg_id,
        "label": raw["label"],
        "summary": raw["summary"],
        "latency_ms": round((time.perf_counter() - started) * 1000, 2),
    }
    required = ("id", "label", "summary", "latency_ms")
    missing = [key for key in required if key not in result]
    if missing:
        raise ValueError(f"result is missing keys: {missing}")
    if result["label"] not in LABELS:
        raise ValueError(f"label {result['label']!r} is not in {LABELS}")
    return result


print("classify_message is ready. Allowed labels:", ", ".join(LABELS))

classify_message is ready. Allowed labels: billing, bug, account, feature, other


## Your First Win

The next cell is the whole move: **paste a message, run one cell, read JSON**.

Keep the sample, or replace `YOUR_TEXT` with your own non-confidential message. Then run the cell.

In [12]:
YOUR_TEXT = "I was charged twice for invoice 1842. Please refund the extra payment."

result = classify_message(YOUR_TEXT, msg_id="demo-001")
print(json.dumps(result, indent=2))

assert set(result) >= {"id", "label", "summary", "latency_ms"}
print("Shape check passed.")

{
  "id": "demo-001",
  "label": "billing",
  "summary": "I was charged twice for invoice 1842. Please refund the extra payment.",
  "latency_ms": 0.01
}
Shape check passed.


In [13]:
YOUR_TEXT = "The app crashes everytime i open report page"

result = classify_message(YOUR_TEXT, msg_id="demo-002")
print(json.dumps(result, indent=2))

assert set(result) >= {"id", "label", "summary", "latency_ms"}
print("Shape check passed.")

{
  "id": "demo-002",
  "label": "bug",
  "summary": "The app crashes everytime i open report page",
  "latency_ms": 0.01
}
Shape check passed.


## Read the Four Fields

That printout is not a chat reply. It is a **record** — something you can store, count, and check.

In [15]:
md(
    f"| Field | Value | What it is |\n"
    f"|-------|-------|------------|\n"
    f"| `id` | `{result['id']}` | Which message this row is about |\n"
    f"| `label` | `{result['label']}` | The bucket the mock chose |\n"
    f"| `summary` | {result['summary']} | A short copy of the input |\n"
    f"| `latency_ms` | {result['latency_ms']} | How long the call took, in milliseconds |\n"
)

| Field | Value | What it is |
|-------|-------|------------|
| `id` | `demo-002` | Which message this row is about |
| `label` | `bug` | The bucket the mock chose |
| `summary` | The app crashes everytime i open report page | A short copy of the input |
| `latency_ms` | 0.01 | How long the call took, in milliseconds |


## The Mock Is Not Magic

Keyword rules pick the **first** matching bucket. That looks confident. It is still a guess.

The next cell is a real support-shaped message. Read it yourself before you run. What label would **you** give it?

In [17]:
tricky = "The refund button on the login page is broken."

tricky_result = classify_message(tricky, msg_id="demo-002")
print(json.dumps(tricky_result, indent=2))
print()
print("A human would likely call this a bug: a broken button.")
print("The mock saw 'refund' first and called it billing.")
print("A plausible label is not the same as a correct label.")

{
  "id": "demo-002",
  "label": "billing",
  "summary": "The refund button on the login page is broken.",
  "latency_ms": 0.01
}

A human would likely call this a bug: a broken button.
The mock saw 'refund' first and called it billing.
A plausible label is not the same as a correct label.


---
## Map the six boxes

The first win is not 'a model demo'. Name the stage each cell just occupied.


## The Six-Box AI Pipeline

Every AI feature you ship — whether it's a chatbot, a classifier, or a document extractor — follows the same six stages:

<div style='display:flex;flex-wrap:wrap;align-items:center;gap:8px;margin:12px 0 16px 0;'>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>1. Ingest</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>2. Prepare</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>3. Prompt</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>4. Call Model</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>5. Parse Output</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>6. Deliver</strong></span>
</div>

| # | Stage | What Happens | Example Failure |
|---|-------|-------------|------------------|
| 1 | **Ingest** | Load raw data (file, API, user input) | File not found, encoding error |
| 2 | **Prepare** | Clean, chunk, or format the data | Text too long for context window |
| 3 | **Prompt** | Build the prompt (system + user + data) | Missing template variable |
| 4 | **Call Model** | Send request to LLM API | API key invalid, rate limit hit |
| 5 | **Parse Output** | Extract structured data from response | JSON parse error, missing field |
| 6 | **Deliver** | Return result to user or downstream system | Wrong format for frontend |

## Define the Pipeline Stages

Let's encode the six stages so we can reference them programmatically.

In [ ]:
PIPELINE_STAGES = {
    1: {"name": "Ingest",       "does": "Load raw data from files, APIs, or user input"},
    2: {"name": "Prepare",      "does": "Clean, chunk, or format data for the model"},
    3: {"name": "Prompt",       "does": "Assemble the system prompt, user message, and context"},
    4: {"name": "Call Model",   "does": "Send the request to the LLM API and wait for a response"},
    5: {"name": "Parse Output", "does": "Extract structured data from the model's raw response"},
    6: {"name": "Deliver",      "does": "Return the result to the user or a downstream system"},
}

# Print the pipeline
header = "| # | Stage | What It Does |\n|---|-------|-------------|\n"
rows = ""
for num, info in PIPELINE_STAGES.items():
    rows += f"| {num} | **{info['name']}** | {info['does']} |\n"

md(header + rows)

## Locate Failures in the Pipeline

The real skill isn't memorizing the six stages — it's **looking at an error and immediately knowing which box it belongs to**. Let's practice with realistic failure scenarios.

In [46]:
# Realistic failure scenarios an AI engineer encounters
failures = [
    {
        "description": "FileNotFoundError: [Errno 2] No such file or directory: 'data/reviews.csv'",
        "stage": 1,
        "explanation": "The data file couldn't be loaded — this is an Ingest failure."
    },
    {
        "description": "The prompt sent 150,000 tokens but the model's context window is 128K.",
        "stage": 2,
        "explanation": "The text wasn't chunked or trimmed properly — this is a Prepare failure."
    },
    {
        "description": "KeyError: 'customer_name' when building the prompt template.",
        "stage": 3,
        "explanation": "A template variable was missing — this is a Prompt failure."
    },
    {
        "description": "openai.AuthenticationError: Invalid API key provided.",
        "stage": 4,
        "explanation": "The API call itself failed — this is a Call Model failure."
    },
    {
        "description": "json.decoder.JSONDecodeError: Expecting ',' — the model returned markdown instead of JSON.",
        "stage": 5,
        "explanation": "The model's response couldn't be parsed — this is a Parse Output failure."
    },
    {
        "description": "The API returns a dict but the frontend expects a flat list of strings.",
        "stage": 6,
        "explanation": "The result format doesn't match what the consumer needs — this is a Deliver failure."
    },
]

In [47]:
def locate_failure(failure):
    """Given a failure dict, return a formatted diagnosis."""
    stage_num = failure["stage"]
    stage_name = PIPELINE_STAGES[stage_num]["name"]
    return (
        f"**Failure:** {failure['description']}\n\n"
        f"📍 **Stage {stage_num} — {stage_name}**\n\n"
        f"*Why:* {failure['explanation']}\n\n---"
    )

In [48]:
# Walk through each failure and locate it in the pipeline
output = "## Failure → Stage Mapping\n\n"
for f in failures:
    output += locate_failure(f) + "\n"

md(output)

## Failure → Stage Mapping

**Failure:** FileNotFoundError: [Errno 2] No such file or directory: 'data/reviews.csv'

📍 **Stage 1 — Ingest**

*Why:* The data file couldn't be loaded — this is an Ingest failure.

---
**Failure:** The prompt sent 150,000 tokens but the model's context window is 128K.

📍 **Stage 2 — Prepare**

*Why:* The text wasn't chunked or trimmed properly — this is a Prepare failure.

---
**Failure:** KeyError: 'customer_name' when building the prompt template.

📍 **Stage 3 — Prompt**

*Why:* A template variable was missing — this is a Prompt failure.

---
**Failure:** openai.AuthenticationError: Invalid API key provided.

📍 **Stage 4 — Call Model**

*Why:* The API call itself failed — this is a Call Model failure.

---
**Failure:** json.decoder.JSONDecodeError: Expecting ',' — the model returned markdown instead of JSON.

📍 **Stage 5 — Parse Output**

*Why:* The model's response couldn't be parsed — this is a Parse Output failure.

---
**Failure:** The API returns a dict but the frontend expects a flat list of strings.

📍 **Stage 6 — Deliver**

*Why:* The result format doesn't match what the consumer needs — this is a Deliver failure.

---


In [ ]:
my_error = "RateLimitError: You exceeded your current quota."
failures[3]['description'] = my_error
output = locate_failure(failures[3])
md(output)

**Failure:** RateLimitError: You exceeded your current quota.

📍 **Stage 4 — Call Model**

*Why:* The API call itself failed — this is a Call Model failure.

---

## Quick Self-Check

Let's verify you can map stages by name. The function below takes a stage name and returns its number and description.

In [11]:
def lookup_stage(name):
    """Look up a pipeline stage by name (case-insensitive)."""
    for num, info in PIPELINE_STAGES.items():
        if info["name"].lower() == name.lower():
            return f"Stage {num}: **{info['name']}** — {info['does']}"
    return f"Unknown stage: '{name}'. Valid stages: {[s['name'] for s in PIPELINE_STAGES.values()]}"

# Test a few lookups
for stage_name in ["Ingest", "Prompt", "Parse Output"]:
    md(lookup_stage(stage_name))

Stage 1: **Ingest** — Load raw data from files, APIs, or user input

Stage 3: **Prompt** — Assemble the system prompt, user message, and context

Stage 5: **Parse Output** — Extract structured data from the model's raw response

## 🎯 Summary

### Key Takeaways

1. **Text in, structured JSON out** — a usable AI feature returns a record (`id`, `label`, `summary`, `latency_ms`), not a chat paragraph
2. **A mock is enough to start** — you can run the path today with no install, no account, and no API key
3. **Plausible is not correct** — a confident JSON label can still be the wrong bucket; the mock is a stand-in, not a judge of quality